# Lab 2: Advanced VarNet — Multi-Slice, Multi-Contrast, Multi-Acceleration

In Lab 1 we explored a single-slice VarNet trained at a single acceleration factor. In this lab we investigate three extensions:

1. **Equispaced vs Random masks** — Does training with equispaced undersampling help when the test data is equispaced?
2. **Multi-acceleration training** — A single model jointly trained at 4x, 5x, and 6x acceleration.
3. **Neighbouring-slice reconstruction** — Feeding 3 adjacent slices (stacked on the coil dimension) so the model can exploit through-plane redundancy.
4. **Joint PD + PDFS contrast reconstruction** — Feeding both contrasts from the same exam so the model can share anatomical information across contrasts.

We compare all variants via SSIM on the test set.

## Part 0 — Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import matplotlib.pyplot as plt
import h5py
import pandas as pd
from pathlib import Path
from collections import defaultdict

from models.varnet import SimpleVarNet
from utils.data import (
    FastMRIKneeDataset, MultiSliceDataset, PairedContrastDataset,
    collate_fn, paired_collate_fn,
)
from utils.transforms import RandomMaskFunc, EquispacedMaskFractionFunc
from utils.metrics import ssim_metric, SSIMLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

# Paths
DATA_PATH = "/gpfs/scratch/johnsp23/DLrecon_lab1/data/knee"
SPLIT_CSV = "/gpfs/scratch/johnsp23/DLrecon_lab1/data/fastMRI_paired_knee.csv"
PRETRAINED = "/gpfs/scratch/johnsp23/DLrecon_lab1/pretrained"
RUNS = os.path.join(os.path.abspath(".."), "runs")

SKIP_SLICES = 4  # skip noisy edge slices for SSIM

In [ ]:
# Helper: load a trained model from checkpoint
def load_model(checkpoint_path, device=device, use_dc=None):
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    cfg = ckpt["config"]
    dc = cfg.get("use_dc", True) if use_dc is None else use_dc
    model = SimpleVarNet(
        num_cascades=cfg.get("num_cascades", 8),
        chans=cfg.get("chans", 18),
        pools=cfg.get("pools", 4),
        sens_chans=cfg.get("sens_chans", 8),
        sens_pools=cfg.get("sens_pools", 4),
        use_dc=dc,
        num_input_slices=cfg.get("num_input_slices", 1),
        num_coils=cfg.get("num_coils", 15),
        num_contrasts=cfg.get("num_contrasts", 1),
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model.to(device), cfg

In [ ]:
# Helper: evaluate a single-output model on a DataLoader, return mean SSIM
@torch.no_grad()
def evaluate(model, loader, device=device):
    model.eval()
    volume_ssims = defaultdict(list)
    for batch in loader:
        masked_kspace, mask, target, max_values, fnames, slice_nums, num_lf = batch
        masked_kspace = masked_kspace.to(device)
        mask = mask.to(device)
        target = target.to(device)
        output = model(masked_kspace, mask, num_low_frequencies=int(num_lf[0]))
        for i in range(output.shape[0]):
            if slice_nums[i] < SKIP_SLICES:
                continue
            ssim_val = ssim_metric(output[i], target[i], max_value=max_values[i])
            volume_ssims[fnames[i]].append(ssim_val)
    all_ssims = [v for vals in volume_ssims.values() for v in vals]
    return np.mean(all_ssims), np.std(all_ssims), volume_ssims


# Helper: evaluate a paired (dual-output) model, return mean SSIM per contrast
@torch.no_grad()
def evaluate_paired(model, loader, device=device):
    model.eval()
    ssims_pd, ssims_pdfs = [], []
    for batch in loader:
        (masked_kspace, mask, target_pd, target_pdfs,
         max_values, pd_fnames, pdfs_fnames, slice_nums, num_lf) = batch
        masked_kspace = masked_kspace.to(device)
        mask = mask.to(device)
        target_pd = target_pd.to(device)
        target_pdfs = target_pdfs.to(device)
        outputs = model(masked_kspace, mask, num_low_frequencies=int(num_lf[0]))
        for i in range(outputs[0].shape[0]):
            if slice_nums[i] < SKIP_SLICES:
                continue
            ssims_pd.append(ssim_metric(outputs[0][i], target_pd[i], max_value=max_values[i]))
            ssims_pdfs.append(ssim_metric(outputs[1][i], target_pdfs[i], max_value=max_values[i]))
    return {
        "pd": (np.mean(ssims_pd), np.std(ssims_pd)),
        "pdfs": (np.mean(ssims_pdfs), np.std(ssims_pdfs)),
        "combined": (np.mean(ssims_pd + ssims_pdfs), np.std(ssims_pd + ssims_pdfs)),
    }

In [ ]:
# Helper: visualize ground truth, reconstruction, and error map
def show_recon(gt, recon, ssim_val, title=""):
    error = np.abs(recon - gt)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(gt, cmap="gray"); axes[0].set_title("Ground truth"); axes[0].axis("off")
    axes[1].imshow(recon, cmap="gray"); axes[1].set_title(f"Reconstruction (SSIM={ssim_val:.4f})"); axes[1].axis("off")
    axes[2].imshow(error * 2, cmap="hot"); axes[2].set_title("Error (2x)"); axes[2].axis("off")
    if title:
        fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

---
## Part 1 — Equispaced vs Random Mask Training

In Lab 1, we trained a model with **random** undersampling masks and tested it on both random and equispaced masks. Now we have a model trained specifically with **equispaced** masks. Does mask-matched training improve SSIM on equispaced test data?

We compare:
- **Random-trained model** evaluated on equispaced 4x test masks
- **Equispaced-trained model** evaluated on equispaced 4x test masks

In [ ]:
# Load both models
random4x_model, random4x_cfg = load_model(os.path.join(PRETRAINED, "varnet_random4x.pt"))
equi4x_model, equi4x_cfg = load_model(os.path.join(RUNS, "equi4x", "best.pt"))

print(f"Random 4x model: {sum(p.numel() for p in random4x_model.parameters()):,} params")
print(f"Equispaced 4x model: {sum(p.numel() for p in equi4x_model.parameters()):,} params")

In [ ]:
# Build equispaced 4x test set
from torch.utils.data import DataLoader

equi4x_test_ds = FastMRIKneeDataset(
    data_path=DATA_PATH, split_csv=SPLIT_CSV, split="test",
    mask_type="equispaced", center_fractions=[0.08], accelerations=[4],
    use_seed=True,
)
equi4x_test_loader = DataLoader(equi4x_test_ds, batch_size=1, shuffle=False,
                                 num_workers=4, collate_fn=collate_fn, pin_memory=True)
print(f"Equispaced 4x test set: {len(equi4x_test_ds)} slices")

In [ ]:
# Evaluate both models on equispaced 4x test data
print("Evaluating random-trained model on equispaced 4x test masks...")
rand_on_equi_mean, rand_on_equi_std, _ = evaluate(random4x_model, equi4x_test_loader)
print(f"  Random-trained:    SSIM = {rand_on_equi_mean:.4f} +/- {rand_on_equi_std:.4f}")

print("Evaluating equispaced-trained model on equispaced 4x test masks...")
equi_on_equi_mean, equi_on_equi_std, _ = evaluate(equi4x_model, equi4x_test_loader)
print(f"  Equispaced-trained: SSIM = {equi_on_equi_mean:.4f} +/- {equi_on_equi_std:.4f}")

print(f"\nDifference: {equi_on_equi_mean - rand_on_equi_mean:+.4f} SSIM")

In [ ]:
# Visual comparison on an example slice
example_batch = next(iter(equi4x_test_loader))
masked_kspace, mask, target, max_values, fnames, slice_nums, num_lf = example_batch
masked_kspace_d = masked_kspace.to(device)
mask_d = mask.to(device)

with torch.no_grad():
    recon_rand = random4x_model(masked_kspace_d, mask_d, num_low_frequencies=int(num_lf[0])).cpu()
    recon_equi = equi4x_model(masked_kspace_d, mask_d, num_low_frequencies=int(num_lf[0])).cpu()

gt = target[0].numpy()
ssim_rand = ssim_metric(recon_rand[0], target[0], max_value=max_values[0])
ssim_equi = ssim_metric(recon_equi[0], target[0], max_value=max_values[0])

show_recon(gt, recon_rand[0].numpy(), ssim_rand, title="Random-trained on equispaced 4x")
show_recon(gt, recon_equi[0].numpy(), ssim_equi, title="Equispaced-trained on equispaced 4x")

### Question 1 (write your answer below)

Why might training with the same mask pattern used at test time help (or not help) reconstruction quality? Think about what the data consistency step does and how the aliasing patterns differ between random and equispaced masks.

*Your answer here.*

---
## Part 2 — Multi-Acceleration Undersampling

So far, every model has been trained at a single acceleration factor (4x). But in practice, radiologists may want different acceleration factors depending on the clinical scenario. Can a **single model** handle multiple accelerations?

We trained a model with masks randomly drawn from {4x, 5x, 6x} during training. Let's see how it performs at each individual acceleration factor.

### Visualize the three mask densities

In [ ]:
# Visualize random masks at 4x, 5x, and 6x
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for i, (accel, ax) in enumerate(zip([4, 5, 6], axes)):
    mf = RandomMaskFunc(center_fractions=[0.08], accelerations=[accel], seed=42)
    m, nlf = mf(shape=(1, 640, 368, 2))
    mask_1d = m[0, 0, :, 0].numpy()
    sampled = int(mask_1d.sum())
    ax.imshow(mask_1d.reshape(1, -1), cmap="gray", aspect="auto", interpolation="none")
    ax.set_title(f"{accel}x: {sampled}/{len(mask_1d)} lines ({100*sampled/len(mask_1d):.0f}%)")
    ax.set_yticks([])
plt.suptitle("Random undersampling masks at different acceleration factors", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 3 — Neighbouring-Slice Reconstruction

MRI volumes are 3D, but our baseline VarNet reconstructs each 2D slice independently. Adjacent slices share most of the same anatomy, just shifted slightly in the through-plane direction. By feeding **3 adjacent slices** (stacked along the coil dimension: 15 coils x 3 slices = 45 channels), the model can exploit this redundancy.

Key design details:
- **Sensitivity maps** are estimated and normalised independently for each group of 15 physical coils (one group per slice). This avoids mixing spatial content from different slices during RSS normalisation.
- The **cascade U-Net** sees one image per slice (3 images = 6 real/imag channels), so it can learn cross-slice features.
- At the output, only the **center slice's 15 coils** are used for the final RSS combination.

This model was jointly trained at 4x, 5x, and 6x acceleration.

In [ ]:
# Load the multi-slice model
ms_model, ms_cfg = load_model(os.path.join(RUNS, "multislice_random_4x5x6x", "best.pt"))
print(f"Multi-slice model: {sum(p.numel() for p in ms_model.parameters()):,} params")
print(f"  num_input_slices = {ms_cfg.get('num_input_slices')}")
print(f"  accelerations = {ms_cfg.get('accelerations')}")

In [ ]:
# Evaluate multi-slice model at each acceleration factor
ms_results = {}
for accel in [4, 5, 6]:
    ds = MultiSliceDataset(
        data_path=DATA_PATH, split_csv=SPLIT_CSV, split="test",
        mask_type="random", center_fractions=[0.08], accelerations=[accel],
        use_seed=True,
    )
    loader = DataLoader(ds, batch_size=1, shuffle=False,
                        num_workers=4, collate_fn=collate_fn, pin_memory=True)
    mean_ssim, std_ssim, _ = evaluate(ms_model, loader)
    ms_results[accel] = (mean_ssim, std_ssim)
    print(f"  Multi-slice @ {accel}x: SSIM = {mean_ssim:.4f} +/- {std_ssim:.4f}")

### Question 2 (write your answer below)

The multi-slice model uses 3x more input channels (45 vs 15) but the same U-Net architecture inside each cascade. How does this affect the model's ability to learn? What kind of information can adjacent slices provide that a single slice cannot?

*Your answer here.*

---
## Part 4 — Joint PD + PDFS Contrast Reconstruction

In the fastMRI knee dataset, each exam includes both a **PD** and a **PDFS** acquisition of the same anatomy. Since the underlying structure is the same (just the contrast weighting differs), we can feed both contrasts to a single model.

The joint-contrast model receives **6 groups of 15 coils** (3 PD slices + 3 PDFS slices = 90 channels) and produces **two outputs**: the center-slice reconstruction for PD and PDFS separately. As with the multi-slice model, sensitivity maps are estimated per-group (6 independent groups of 15 coils), and the cascade U-Net sees 6 images (12 real/imag channels) — enabling cross-slice AND cross-contrast feature learning.

This model was jointly trained at 4x, 5x, and 6x acceleration.

In [ ]:
# Load the joint-contrast model
jc_model, jc_cfg = load_model(os.path.join(RUNS, "joint_contrast_random_4x5x6x", "best.pt"))
print(f"Joint-contrast model: {sum(p.numel() for p in jc_model.parameters()):,} params")
print(f"  num_input_slices = {jc_cfg.get('num_input_slices')}")
print(f"  num_contrasts = {jc_cfg.get('num_contrasts')}")
print(f"  accelerations = {jc_cfg.get('accelerations')}")

In [ ]:
# Evaluate joint-contrast model at each acceleration factor
jc_results = {}
for accel in [4, 5, 6]:
    ds = PairedContrastDataset(
        data_path=DATA_PATH, split_csv=SPLIT_CSV, split="test",
        mask_type="random", center_fractions=[0.08], accelerations=[accel],
        use_seed=True,
    )
    loader = DataLoader(ds, batch_size=1, shuffle=False,
                        num_workers=4, collate_fn=paired_collate_fn, pin_memory=True)
    result = evaluate_paired(jc_model, loader)
    jc_results[accel] = result
    print(f"  Joint-contrast @ {accel}x:")
    print(f"    PD:   SSIM = {result['pd'][0]:.4f} +/- {result['pd'][1]:.4f}")
    print(f"    PDFS: SSIM = {result['pdfs'][0]:.4f} +/- {result['pdfs'][1]:.4f}")
    print(f"    Avg:  SSIM = {result['combined'][0]:.4f} +/- {result['combined'][1]:.4f}")

In [ ]:
# Visual comparison: joint-contrast model on a paired example at 4x
ds_vis = PairedContrastDataset(
    data_path=DATA_PATH, split_csv=SPLIT_CSV, split="test",
    mask_type="random", center_fractions=[0.08], accelerations=[4],
    use_seed=True,
)
loader_vis = DataLoader(ds_vis, batch_size=1, shuffle=False, collate_fn=paired_collate_fn)

# Get a mid-volume slice (skip edge slices)
for batch in loader_vis:
    (ksp, msk, tgt_pd, tgt_pdfs, mv, pdf, pdfsf, sl, nlf) = batch
    if sl[0] >= 15:  # pick a nice mid-volume slice
        break

with torch.no_grad():
    outputs = jc_model(ksp.to(device), msk.to(device), num_low_frequencies=int(nlf[0]))
    recon_pd = outputs[0].cpu()
    recon_pdfs = outputs[1].cpu()

ssim_pd = ssim_metric(recon_pd[0], tgt_pd[0], max_value=mv[0])
ssim_pdfs = ssim_metric(recon_pdfs[0], tgt_pdfs[0], max_value=mv[0])

show_recon(tgt_pd[0].numpy(), recon_pd[0].numpy(), ssim_pd,
           title=f"Joint-contrast: PD output (slice {sl[0]})")
show_recon(tgt_pdfs[0].numpy(), recon_pdfs[0].numpy(), ssim_pdfs,
           title=f"Joint-contrast: PDFS output (slice {sl[0]})")

### Question 3 (write your answer below)

The joint-contrast model has the same number of cascades and U-Net channels as the single-slice model, but processes 90 input channels instead of 15. What are the potential advantages and disadvantages of this approach? Can you think of clinical scenarios where jointly reconstructing PD and PDFS would be especially beneficial?

*Your answer here.*

---
## Part 5 — SSIM Comparison Across All Models

For a fair comparison, all three models were trained with the same acceleration mix (4x/5x/6x random masks). The only difference is the input representation:

| Model | Input | Groups | U-Net channels |
|-------|-------|--------|---------------|
| **Baseline** | 1 slice, 1 contrast (15 coils) | 1 | 2 |
| **Multi-slice** | 3 slices, 1 contrast (45 coils) | 3 | 6 |
| **Joint-contrast** | 3 slices, 2 contrasts (90 coils) | 6 | 12 |

In [ ]:
# Load and evaluate single-slice baseline (trained at 4x/5x/6x) at each acceleration
baseline_model, baseline_cfg = load_model(os.path.join(RUNS, "baseline_random_4x5x6x", "best.pt"))
print(f"Baseline model: {sum(p.numel() for p in baseline_model.parameters()):,} params")

baseline_results = {}
for accel in [4, 5, 6]:
    ds = FastMRIKneeDataset(
        data_path=DATA_PATH, split_csv=SPLIT_CSV, split="test",
        mask_type="random", center_fractions=[0.08], accelerations=[accel],
        use_seed=True,
    )
    loader = DataLoader(ds, batch_size=1, shuffle=False,
                        num_workers=4, collate_fn=collate_fn, pin_memory=True)
    mean_ssim, std_ssim, _ = evaluate(baseline_model, loader)
    baseline_results[accel] = (mean_ssim, std_ssim)
    print(f"  Baseline (4x/5x/6x) @ {accel}x: SSIM = {mean_ssim:.4f} +/- {std_ssim:.4f}")

In [ ]:
# Summary table
print("=" * 70)
print(f"{'Model':<30s} {'4x SSIM':>12s} {'5x SSIM':>12s} {'6x SSIM':>12s}")
print("=" * 70)

print(f"{'Baseline (single-slice)':<30s}", end="")
for accel in [4, 5, 6]:
    m, s = baseline_results[accel]
    print(f" {m:.4f}+/-{s:.3f}", end="")
print()

print(f"{'Multi-slice (3-slice)':<30s}", end="")
for accel in [4, 5, 6]:
    m, s = ms_results[accel]
    print(f" {m:.4f}+/-{s:.3f}", end="")
print()

print(f"{'Joint-contrast (PD+PDFS)':<30s}", end="")
for accel in [4, 5, 6]:
    m, s = jc_results[accel]["combined"]
    print(f" {m:.4f}+/-{s:.3f}", end="")
print()

print("=" * 70)

In [ ]:
# Bar chart comparison
accels = [4, 5, 6]
x = np.arange(len(accels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))

baseline_means = [baseline_results[a][0] for a in accels]
ms_means = [ms_results[a][0] for a in accels]
jc_means = [jc_results[a]["combined"][0] for a in accels]

bars1 = ax.bar(x - width, baseline_means, width, label="Baseline (single-slice)")
bars2 = ax.bar(x, ms_means, width, label="Multi-slice (3-slice)")
bars3 = ax.bar(x + width, jc_means, width, label="Joint-contrast (PD+PDFS)")

ax.set_xlabel("Acceleration Factor")
ax.set_ylabel("SSIM")
ax.set_title("Test SSIM by Model and Acceleration Factor (all trained at 4x/5x/6x)")
ax.set_xticks(x)
ax.set_xticklabels([f"{a}x" for a in accels])
ax.legend()
ax.set_ylim(bottom=0.8)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

---
### Question 4 (write your answer below)

Looking at the summary table and bar chart (all three models trained on the same 4x/5x/6x acceleration mix):

1. Does the multi-slice model improve over the single-slice baseline? At which acceleration factors is the improvement largest, and why?
2. Does joint-contrast reconstruction provide additional benefit beyond multi-slice? What might explain the relative performance of PD vs PDFS reconstruction?
3. All three models have the same number of cascades (12) and U-Net base channels (24), but the U-Net input channels differ (2 vs 6 vs 12). What are the trade-offs of increasing input channels while keeping the architecture fixed?

*Your answer here.*

---
## Summary

In this lab, we explored several extensions to the baseline VarNet:

| Extension | Key idea | Trade-off |
|-----------|----------|-----------|
| **Mask-matched training** | Train with the same mask pattern used at test time | Requires knowing the test-time mask type in advance |
| **Multi-acceleration** | Randomly sample from {4x, 5x, 6x} during training | Single model handles multiple scan protocols |
| **Multi-slice** | Stack 3 adjacent slices (45 coils) | Exploits through-plane redundancy; 3x more memory per sample |
| **Joint-contrast** | Stack PD + PDFS (90 coils) | Shared anatomy across contrasts; requires paired acquisitions |

All models use the same VarNet architecture (12 cascades, 24-channel U-Net) — the only difference is what information is provided as input. This demonstrates that **the choice of input representation** can be as important as the model architecture itself.